In [1]:
from pathlib import Path
import pandas as pd

current = Path.cwd()

project_root = next(
    (folder for folder in [current, *current.parents]
     if (folder / "data" / "processed" / "groundwater_cleaned.csv").exists()),
    None
)

if project_root is None:
    raise FileNotFoundError("Could not find the cleaned dataset.")

data_path = project_root / "data" / "processed" / "groundwater_cleaned.csv"

df = pd.read_csv(data_path)

print("Project folder:", project_root)
print("Dataset shape:", df.shape)

display(df.head())

Project folder: D:\GroundWatch
Dataset shape: (123467, 24)


,SlNo,Station,Agency,State LGD Code,State,District LGD Code,District,Tehsil,Block,Village,...,SubSubtributary,Local River,Latitude,Longitude,RL_MSL,Data Acquisition Time,Groundwater Level Quarterly Manual (meter),Station_ID,Source,Anomaly_Flag
0,1,Abbigere,CGWB,29,Karnataka,543,Koppal,KOPPAL,KOPPAL,Abbigere,...,-,-,15.430556,76.114167,NaN,2012-01-05 06:00:00,2.23,Abbigere_Koppal_15.43055556_76.11416667,Historical,False
1,2,Abhyath Mangale,CGWB,29,Karnataka,541,Kodagu,SOMVARPET,SOMVARPET,Abhyath Mangale,...,-,-,12.330556,75.856111,NaN,2012-11-05 06:00:00,3.35,Abhyath Mangale_Kodagu_12.33055556_75.85611111,Historical,False
2,3,Abhyath Mangale,CGWB,29,Karnataka,541,Kodagu,SOMVARPET,SOMVARPET,Abhyath Mangale,...,-,-,12.330556,75.856111,NaN,2013-01-05 06:00:00,3.74,Abhyath Mangale_Kodagu_12.33055556_75.85611111,Historical,False
3,4,Abhyath Mangale,CGWB,29,Karnataka,541,Kodagu,SOMVARPET,SOMVARPET,Abhyath Mangale,...,-,-,12.330556,75.856111,NaN,2013-08-25 06:00:00,1.82,Abhyath Mangale_Kodagu_12.33055556_75.85611111,Historical,False
4,5,Abhyath Mangale,CGWB,29,Karnataka,541,Kodagu,SOMVARPET,SOMVARPET,Abhyath Mangale,...,-,-,12.330556,75.856111,NaN,2013-11-05 06:00:00,4.04,Abhyath Mangale_Kodagu_12.33055556_75.85611111,Historical,False


In [2]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

Column names:
['SlNo', 'Station', 'Agency', 'State LGD Code', 'State', 'District LGD Code', 'District', 'Tehsil', 'Block', 'Village', 'River', 'Basin', 'Tributary', 'Subtributary', 'SubSubtributary', 'Local River', 'Latitude', 'Longitude', 'RL_MSL', 'Data Acquisition Time', 'Groundwater Level Quarterly Manual (meter)', 'Station_ID', 'Source', 'Anomaly_Flag']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123467 entries, 0 to 123466
Data columns (total 24 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   SlNo                                        123467 non-null  int64  
 1   Station                                     123467 non-null  object 
 2   Agency                                      123467 non-null  object 
 3   State LGD Code                              123467 non-null  int64  
 4   State                                       123467 non-nu

In [3]:
print("Number of unique stations:", df["Station_ID"].nunique())

print("\nDate examples:")
print(df["Data Acquisition Time"].head(10))

print("\nRecords per station:")
print(df["Station_ID"].value_counts().describe())

print("\nAnomaly flag counts:")
print(df["Anomaly_Flag"].value_counts())

Number of unique stations: 3073

Date examples:
0    2012-01-05 06:00:00
1    2012-11-05 06:00:00
2    2013-01-05 06:00:00
3    2013-08-25 06:00:00
4    2013-11-05 06:00:00
5    2014-01-05 06:00:00
6    2014-05-15 06:00:00
7    2014-08-25 06:00:00
8    2014-11-05 06:00:00
9    2015-01-05 06:00:00
Name: Data Acquisition Time, dtype: object

Records per station:
count    3073.000000
mean       40.178002
std        32.684660
min         1.000000
25%        12.000000
50%        36.000000
75%        63.000000
max       109.000000
Name: count, dtype: float64

Anomaly flag counts:
Anomaly_Flag
False    123391
True         76
Name: count, dtype: int64


In [4]:
model_df = df.copy()

model_df["Data Acquisition Time"] = pd.to_datetime(
    model_df["Data Acquisition Time"],
    errors="coerce"
)

print("Earliest date:", model_df["Data Acquisition Time"].min())
print("Latest date:", model_df["Data Acquisition Time"].max())

print(
    "Dates that could not be converted:",
    model_df["Data Acquisition Time"].isna().sum()
)


Earliest date: 1996-01-05 06:00:00
Latest date: 2024-08-30 10:00:00
Dates that could not be converted: 0


In [5]:
# Sort readings by station and date
model_df = model_df.sort_values(
    by=["Station_ID", "Data Acquisition Time"]
).copy()

# Previous reading from the same station
model_df["Previous_Groundwater_Level"] = (
    model_df.groupby("Station_ID")[
        "Groundwater Level Quarterly Manual (meter)"
    ].shift(1)
)

print("Feature created successfully!")

display(
    model_df[
        [
            "Station_ID",
            "Data Acquisition Time",
            "Groundwater Level Quarterly Manual (meter)",
            "Previous_Groundwater_Level"
        ]
    ].head(10)
)

Feature created successfully!


,Station_ID,Data Acquisition Time,Groundwater Level Quarterly Manual (meter),Previous_Groundwater_Level
104850,ADUGODI_Bangalore Urban_12.94361111_77.60916667,2023-01-05 00:00:00,4.30,NaN
104851,ADUGODI_Bangalore Urban_12.94361111_77.60916667,2023-05-25 00:00:00,4.30,4.30
104852,ADUGODI_Bangalore Urban_12.94361111_77.60916667,2023-08-25 00:00:00,4.45,4.30
104853,ADUGODI_Bangalore Urban_12.94361111_77.60916667,2023-11-05 00:00:00,5.10,4.45
104854,ADUGODI_Bangalore Urban_12.94361111_77.60916667,2024-01-05 00:00:00,13.97,5.10
0,Abbigere_Koppal_15.43055556_76.11416667,2012-01-05 06:00:00,2.23,NaN
1,Abhyath Mangale_Kodagu_12.33055556_75.85611111,2012-11-05 06:00:00,3.35,NaN
2,Abhyath Mangale_Kodagu_12.33055556_75.85611111,2013-01-05 06:00:00,3.74,3.35
3,Abhyath Mangale_Kodagu_12.33055556_75.85611111,2013-08-25 06:00:00,1.82,3.74
4,Abhyath Mangale_Kodagu_12.33055556_75.85611111,2013-11-05 06:00:00,4.04,1.82


In [6]:
model_df["Previous_2_Groundwater_Level"] = (
    model_df.groupby("Station_ID")[
        "Groundwater Level Quarterly Manual (meter)"
    ].shift(2)
)

display(
    model_df[
        [
            "Station_ID",
            "Data Acquisition Time",
            "Groundwater Level Quarterly Manual (meter)",
            "Previous_Groundwater_Level",
            "Previous_2_Groundwater_Level"
        ]
    ].head(10)
)

,Station_ID,Data Acquisition Time,Groundwater Level Quarterly Manual (meter),Previous_Groundwater_Level,Previous_2_Groundwater_Level
104850,ADUGODI_Bangalore Urban_12.94361111_77.60916667,2023-01-05 00:00:00,4.30,NaN,NaN
104851,ADUGODI_Bangalore Urban_12.94361111_77.60916667,2023-05-25 00:00:00,4.30,4.30,NaN
104852,ADUGODI_Bangalore Urban_12.94361111_77.60916667,2023-08-25 00:00:00,4.45,4.30,4.30
104853,ADUGODI_Bangalore Urban_12.94361111_77.60916667,2023-11-05 00:00:00,5.10,4.45,4.30
104854,ADUGODI_Bangalore Urban_12.94361111_77.60916667,2024-01-05 00:00:00,13.97,5.10,4.45
0,Abbigere_Koppal_15.43055556_76.11416667,2012-01-05 06:00:00,2.23,NaN,NaN
1,Abhyath Mangale_Kodagu_12.33055556_75.85611111,2012-11-05 06:00:00,3.35,NaN,NaN
2,Abhyath Mangale_Kodagu_12.33055556_75.85611111,2013-01-05 06:00:00,3.74,3.35,NaN
3,Abhyath Mangale_Kodagu_12.33055556_75.85611111,2013-08-25 06:00:00,1.82,3.74,3.35
4,Abhyath Mangale_Kodagu_12.33055556_75.85611111,2013-11-05 06:00:00,4.04,1.82,3.74


In [7]:
model_df["Previous_3_Groundwater_Level"] = (
    model_df.groupby("Station_ID")[
        "Groundwater Level Quarterly Manual (meter)"
    ].shift(3)
)

print("Third historical feature created!")

display(
    model_df[
        [
            "Station_ID",
            "Groundwater Level Quarterly Manual (meter)",
            "Previous_Groundwater_Level",
            "Previous_2_Groundwater_Level",
            "Previous_3_Groundwater_Level"
        ]
    ].head(10)
)

Third historical feature created!


,Station_ID,Groundwater Level Quarterly Manual (meter),Previous_Groundwater_Level,Previous_2_Groundwater_Level,Previous_3_Groundwater_Level
104850,ADUGODI_Bangalore Urban_12.94361111_77.60916667,4.30,NaN,NaN,NaN
104851,ADUGODI_Bangalore Urban_12.94361111_77.60916667,4.30,4.30,NaN,NaN
104852,ADUGODI_Bangalore Urban_12.94361111_77.60916667,4.45,4.30,4.30,NaN
104853,ADUGODI_Bangalore Urban_12.94361111_77.60916667,5.10,4.45,4.30,4.30
104854,ADUGODI_Bangalore Urban_12.94361111_77.60916667,13.97,5.10,4.45,4.30
0,Abbigere_Koppal_15.43055556_76.11416667,2.23,NaN,NaN,NaN
1,Abhyath Mangale_Kodagu_12.33055556_75.85611111,3.35,NaN,NaN,NaN
2,Abhyath Mangale_Kodagu_12.33055556_75.85611111,3.74,3.35,NaN,NaN
3,Abhyath Mangale_Kodagu_12.33055556_75.85611111,1.82,3.74,3.35,NaN
4,Abhyath Mangale_Kodagu_12.33055556_75.85611111,4.04,1.82,3.74,3.35


In [8]:
model_df["Month"] = model_df["Data Acquisition Time"].dt.month

print("Month feature created!")

display(
    model_df[
        [
            "Station_ID",
            "Data Acquisition Time",
            "Month"
        ]
    ].head(10)
)

Month feature created!


,Station_ID,Data Acquisition Time,Month
104850,ADUGODI_Bangalore Urban_12.94361111_77.60916667,2023-01-05 00:00:00,1
104851,ADUGODI_Bangalore Urban_12.94361111_77.60916667,2023-05-25 00:00:00,5
104852,ADUGODI_Bangalore Urban_12.94361111_77.60916667,2023-08-25 00:00:00,8
104853,ADUGODI_Bangalore Urban_12.94361111_77.60916667,2023-11-05 00:00:00,11
104854,ADUGODI_Bangalore Urban_12.94361111_77.60916667,2024-01-05 00:00:00,1
0,Abbigere_Koppal_15.43055556_76.11416667,2012-01-05 06:00:00,1
1,Abhyath Mangale_Kodagu_12.33055556_75.85611111,2012-11-05 06:00:00,11
2,Abhyath Mangale_Kodagu_12.33055556_75.85611111,2013-01-05 06:00:00,1
3,Abhyath Mangale_Kodagu_12.33055556_75.85611111,2013-08-25 06:00:00,8
4,Abhyath Mangale_Kodagu_12.33055556_75.85611111,2013-11-05 06:00:00,11


In [9]:
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Post-Monsoon"


model_df["Season"] = model_df["Month"].apply(get_season)

display(
    model_df[
        ["Data Acquisition Time", "Month", "Season"]
    ].head(10)
)


,Data Acquisition Time,Month,Season
104850,2023-01-05 00:00:00,1,Winter
104851,2023-05-25 00:00:00,5,Summer
104852,2023-08-25 00:00:00,8,Monsoon
104853,2023-11-05 00:00:00,11,Post-Monsoon
104854,2024-01-05 00:00:00,1,Winter
0,2012-01-05 06:00:00,1,Winter
1,2012-11-05 06:00:00,11,Post-Monsoon
2,2013-01-05 06:00:00,1,Winter
3,2013-08-25 06:00:00,8,Monsoon
4,2013-11-05 06:00:00,11,Post-Monsoon


In [10]:
print("Season-wise record distribution:")

print(model_df["Season"].value_counts())

print("\nSeason percentages:")

print(
    model_df["Season"].value_counts(normalize=True) * 100
)

Season-wise record distribution:
Season
Post-Monsoon    32603
Winter          32477
Monsoon         30423
Summer          27964
Name: count, dtype: int64

Season percentages:
Season
Post-Monsoon    26.406246
Winter          26.304195
Monsoon         24.640592
Summer          22.648967
Name: proportion, dtype: float64


In [11]:
model_df["Year"] = model_df["Data Acquisition Time"].dt.year

print("Year feature created!")

display(
    model_df[
        ["Data Acquisition Time", "Month", "Season", "Year"]
    ].head(10)
)

Year feature created!


,Data Acquisition Time,Month,Season,Year
104850,2023-01-05 00:00:00,1,Winter,2023
104851,2023-05-25 00:00:00,5,Summer,2023
104852,2023-08-25 00:00:00,8,Monsoon,2023
104853,2023-11-05 00:00:00,11,Post-Monsoon,2023
104854,2024-01-05 00:00:00,1,Winter,2024
0,2012-01-05 06:00:00,1,Winter,2012
1,2012-11-05 06:00:00,11,Post-Monsoon,2012
2,2013-01-05 06:00:00,1,Winter,2013
3,2013-08-25 06:00:00,8,Monsoon,2013
4,2013-11-05 06:00:00,11,Post-Monsoon,2013


In [12]:
print("Year-wise record distribution:")

print(
    model_df["Year"].value_counts().sort_index()
)

Year-wise record distribution:
Year
1996    3056
1997    3112
1998    3011
1999    3483
2000    3562
2001    3606
2002    3410
2003    2457
2004    3076
2005    3418
2006    3587
2007    4040
2008    4090
2009    4044
2010    3907
2011    3961
2012    3216
2013    6124
2014    6418
2015    6546
2016    5430
2017    6355
2018    6244
2019    5632
2020    2996
2021    4410
2022    5701
2023    6089
2024    2486
Name: count, dtype: int64


In [13]:
model_df["Rolling_Average_3"] = (
    model_df.groupby("Station_ID")[
        "Groundwater Level Quarterly Manual (meter)"
    ]
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=3).mean())
)

print("Rolling average feature created!")

display(
    model_df[
        [
            "Station_ID",
            "Groundwater Level Quarterly Manual (meter)",
            "Previous_Groundwater_Level",
            "Previous_2_Groundwater_Level",
            "Previous_3_Groundwater_Level",
            "Rolling_Average_3"
        ]
    ].head(10)
)

Rolling average feature created!


,Station_ID,Groundwater Level Quarterly Manual (meter),Previous_Groundwater_Level,Previous_2_Groundwater_Level,Previous_3_Groundwater_Level,Rolling_Average_3
104850,ADUGODI_Bangalore Urban_12.94361111_77.60916667,4.30,NaN,NaN,NaN,NaN
104851,ADUGODI_Bangalore Urban_12.94361111_77.60916667,4.30,4.30,NaN,NaN,NaN
104852,ADUGODI_Bangalore Urban_12.94361111_77.60916667,4.45,4.30,4.30,NaN,NaN
104853,ADUGODI_Bangalore Urban_12.94361111_77.60916667,5.10,4.45,4.30,4.30,4.350000
104854,ADUGODI_Bangalore Urban_12.94361111_77.60916667,13.97,5.10,4.45,4.30,4.616667
0,Abbigere_Koppal_15.43055556_76.11416667,2.23,NaN,NaN,NaN,NaN
1,Abhyath Mangale_Kodagu_12.33055556_75.85611111,3.35,NaN,NaN,NaN,NaN
2,Abhyath Mangale_Kodagu_12.33055556_75.85611111,3.74,3.35,NaN,NaN,NaN
3,Abhyath Mangale_Kodagu_12.33055556_75.85611111,1.82,3.74,3.35,NaN,NaN
4,Abhyath Mangale_Kodagu_12.33055556_75.85611111,4.04,1.82,3.74,3.35,2.970000


In [14]:
feature_columns = [
    "Previous_Groundwater_Level",
    "Previous_2_Groundwater_Level",
    "Previous_3_Groundwater_Level",
    "Month",
    "Season",
    "Year",
    "Rolling_Average_3"
]

print("Missing values in engineered features:")

print(model_df[feature_columns].isnull().sum())

Missing values in engineered features:
Previous_Groundwater_Level      3073
Previous_2_Groundwater_Level    5922
Previous_3_Groundwater_Level    8635
Month                              0
Season                             0
Year                               0
Rolling_Average_3               8635
dtype: int64


In [15]:
complete_history = model_df.dropna(
    subset=[
        "Previous_Groundwater_Level",
        "Previous_2_Groundwater_Level",
        "Previous_3_Groundwater_Level"
    ]
)

print("Total records:", len(model_df))
print("Records with complete history:", len(complete_history))
print("Records without complete history:", len(model_df) - len(complete_history))

Total records: 123467
Records with complete history: 114832
Records without complete history: 8635


In [16]:
print(
    "Records with rolling average:",
    model_df["Rolling_Average_3"].notna().sum()
)

print(
    "Records without rolling average:",
    model_df["Rolling_Average_3"].isna().sum()
)

Records with rolling average: 114832
Records without rolling average: 8635


In [17]:
station_counts = model_df.groupby("Station_ID").size()

print("Stations with at least 4 readings:",
      (station_counts >= 4).sum())

print("Stations with fewer than 4 readings:",
      (station_counts < 4).sum())

print("Total stations:", station_counts.count())

Stations with at least 4 readings: 2640
Stations with fewer than 4 readings: 433
Total stations: 3073


In [18]:
eligible_stations = station_counts[station_counts >= 4]

print("Eligible station history summary:")
print(eligible_stations.describe())

print("\nStations with at least:")
for n in [4, 8, 12, 20, 30]:
    print(f"{n} readings: {(station_counts >= n).sum()}")

Eligible station history summary:
count    2640.000000
mean       46.496970
std        30.983708
min         4.000000
25%        23.000000
50%        40.000000
75%        70.000000
max       109.000000
dtype: float64

Stations with at least:
4 readings: 2640
8 readings: 2477
12 readings: 2318
20 readings: 2041
30 readings: 1784


In [20]:
# Define the target column
target = "Groundwater Level Quarterly Manual (meter)"

# Calculate the historical coverage of each station
station_history = model_df.groupby("Station_ID").agg(
    reading_count=(target, "count"),
    first_reading=("Data Acquisition Time", "min"),
    last_reading=("Data Acquisition Time", "max")
)

# Calculate the time coverage in years
station_history["coverage_years"] = (
    station_history["last_reading"] - station_history["first_reading"]
).dt.days / 365.25

# Display overall time coverage statistics
print("Station Time Coverage Summary:")
print(station_history["coverage_years"].describe())

# Count stations with at least 12 readings
print("\nStations with at least 12 readings:")
print((station_history["reading_count"] >= 12).sum())

# Count stations with at least 12 readings
# and at least 3 years of historical coverage
print("\nStations with at least 12 readings and 3+ years of coverage:")
print(
    (
        (station_history["reading_count"] >= 12) &
        (station_history["coverage_years"] >= 3)
    ).sum()
)

Station Time Coverage Summary:
count    3073.000000
mean       13.104923
std         9.485754
min         0.000000
25%         4.635181
50%        11.162218
75%        21.998631
max        28.651608
Name: coverage_years, dtype: float64

Stations with at least 12 readings:
2318

Stations with at least 12 readings and 3+ years of coverage:
2308


In [21]:
# Select stations with at least 12 readings
# and at least 3 years of historical coverage

eligible_station_ids = station_history[
    (station_history["reading_count"] >= 12) &
    (station_history["coverage_years"] >= 3)
].index

# Create modelling dataset using eligible stations
model_data = model_df[
    model_df["Station_ID"].isin(eligible_station_ids)
].copy()

print("Total eligible stations:", len(eligible_station_ids))
print("Total records selected:", len(model_data))

print("\nMissing values in engineered features:")
print(
    model_data[
        [
            "Previous_Groundwater_Level",
            "Previous_2_Groundwater_Level",
            "Previous_3_Groundwater_Level",
            "Rolling_Average_3"
        ]
    ].isnull().sum()
)

Total eligible stations: 2308
Total records selected: 120234

Missing values in engineered features:
Previous_Groundwater_Level      2308
Previous_2_Groundwater_Level    4616
Previous_3_Groundwater_Level    6924
Rolling_Average_3               6924
dtype: int64


In [22]:
# Required features for model training
feature_columns = [
    "Previous_Groundwater_Level",
    "Previous_2_Groundwater_Level",
    "Previous_3_Groundwater_Level",
    "Rolling_Average_3",
    "Month",
    "Year"
]

# Remove rows with missing required features
final_model_data = model_data.dropna(
    subset=feature_columns + [target]
).copy()

print("Original selected records:", len(model_data))
print("Records after removing incomplete history:", len(final_model_data))
print("Records removed:", len(model_data) - len(final_model_data))

print("\nRemaining missing values:")
print(final_model_data[feature_columns + [target]].isnull().sum())

print("\nNumber of stations remaining:",
      final_model_data["Station_ID"].nunique())

Original selected records: 120234
Records after removing incomplete history: 113310
Records removed: 6924

Remaining missing values:
Previous_Groundwater_Level                    0
Previous_2_Groundwater_Level                  0
Previous_3_Groundwater_Level                  0
Rolling_Average_3                             0
Month                                         0
Year                                          0
Groundwater Level Quarterly Manual (meter)    0
dtype: int64

Number of stations remaining: 2308


In [23]:
print("FEATURE COLUMNS:")
for column in feature_columns:
    print("-", column)

print("\nTARGET COLUMN:")
print(target)

print("\nFINAL DATASET SHAPE:")
print(final_model_data.shape)

print("\nDATA TYPES:")
print(final_model_data[feature_columns + [target]].dtypes)

print("\nSAMPLE DATA:")
display(final_model_data[feature_columns + [target]].head())

FEATURE COLUMNS:
- Previous_Groundwater_Level
- Previous_2_Groundwater_Level
- Previous_3_Groundwater_Level
- Rolling_Average_3
- Month
- Year

TARGET COLUMN:
Groundwater Level Quarterly Manual (meter)

FINAL DATASET SHAPE:
(113310, 31)

DATA TYPES:
Previous_Groundwater_Level                    float64
Previous_2_Groundwater_Level                  float64
Previous_3_Groundwater_Level                  float64
Rolling_Average_3                             float64
Month                                           int32
Year                                            int32
Groundwater Level Quarterly Manual (meter)    float64
dtype: object

SAMPLE DATA:


,Previous_Groundwater_Level,Previous_2_Groundwater_Level,Previous_3_Groundwater_Level,Rolling_Average_3,Month,Year,Groundwater Level Quarterly Manual (meter)
4,1.82,3.74,3.35,2.970000,11,2013,4.04
5,4.04,1.82,3.74,3.200000,1,2014,2.84
6,2.84,4.04,1.82,2.900000,5,2014,3.49
7,3.49,2.84,4.04,3.456667,8,2014,2.14
8,2.14,3.49,2.84,2.823333,11,2014,2.04


In [24]:
# Display input features
print("FEATURE COLUMNS:")
for column in feature_columns:
    print("-", column)

# Display target
print("\nTARGET COLUMN:")
print(target)

# Check dataset dimensions
print("\nFINAL DATASET SHAPE:")
print(final_model_data.shape)

# Check data types
print("\nDATA TYPES:")
print(final_model_data[feature_columns + [target]].dtypes)

# Check sample records
print("\nSAMPLE DATA:")
display(final_model_data[feature_columns + [target]].head())

# Check for missing values
print("\nMISSING VALUES:")
print(final_model_data[feature_columns + [target]].isnull().sum())

FEATURE COLUMNS:
- Previous_Groundwater_Level
- Previous_2_Groundwater_Level
- Previous_3_Groundwater_Level
- Rolling_Average_3
- Month
- Year

TARGET COLUMN:
Groundwater Level Quarterly Manual (meter)

FINAL DATASET SHAPE:
(113310, 31)

DATA TYPES:
Previous_Groundwater_Level                    float64
Previous_2_Groundwater_Level                  float64
Previous_3_Groundwater_Level                  float64
Rolling_Average_3                             float64
Month                                           int32
Year                                            int32
Groundwater Level Quarterly Manual (meter)    float64
dtype: object

SAMPLE DATA:


,Previous_Groundwater_Level,Previous_2_Groundwater_Level,Previous_3_Groundwater_Level,Rolling_Average_3,Month,Year,Groundwater Level Quarterly Manual (meter)
4,1.82,3.74,3.35,2.970000,11,2013,4.04
5,4.04,1.82,3.74,3.200000,1,2014,2.84
6,2.84,4.04,1.82,2.900000,5,2014,3.49
7,3.49,2.84,4.04,3.456667,8,2014,2.14
8,2.14,3.49,2.84,2.823333,11,2014,2.04



MISSING VALUES:
Previous_Groundwater_Level                    0
Previous_2_Groundwater_Level                  0
Previous_3_Groundwater_Level                  0
Rolling_Average_3                             0
Month                                         0
Year                                          0
Groundwater Level Quarterly Manual (meter)    0
dtype: int64


In [25]:
final_model_data = final_model_data.sort_values(
    by="Data Acquisition Time"
).copy()

final_model_data = final_model_data.reset_index(drop=True)

print("First observation:")
print(final_model_data["Data Acquisition Time"].min())

print("\nLast observation:")
print(final_model_data["Data Acquisition Time"].max())

print("\nTotal records:", len(final_model_data))


First observation:
1996-11-05 06:00:00

Last observation:
2024-08-30 10:00:00

Total records: 113310


In [26]:
from pathlib import Path

output_path = Path.cwd() / "data" / "processed" / "groundwater_features.csv"

output_path.parent.mkdir(parents=True, exist_ok=True)

final_model_data.to_csv(output_path, index=False)

print("Dataset saved successfully!")
print("File location:", output_path)
print("Rows:", len(final_model_data))


Dataset saved successfully!
File location: D:\GroundWatch\notebooks\data\processed\groundwater_features.csv
Rows: 113310


In [27]:
from pathlib import Path

repo_root = Path.cwd()

while repo_root.name != "GroundWatch" and repo_root.parent != repo_root:
    repo_root = repo_root.parent

output_path = repo_root / "data" / "processed" / "groundwater_features.csv"

output_path.parent.mkdir(parents=True, exist_ok=True)

final_model_data.to_csv(output_path, index=False)

print("Dataset saved successfully!")
print("Correct location:", output_path)
print("Rows:", len(final_model_data))

Dataset saved successfully!
Correct location: D:\GroundWatch\data\processed\groundwater_features.csv
Rows: 113310
